# Muestreo por usuario con cuota por actividad

[04_unidad_de_analisis.ipynb](04_unidad_de_analisis.ipynb) mostró que el muestreo por `(categoría, rating)` rompe el historial de usuario (0,5% con ≥2 reviews). Acá se prueba `user_quota_sample_reviews`:

1. filtro por hash de `user_id` (consistente entre categorías),
2. cuota de usuarios por tier de actividad (`1`, `2-4`, `5+` reviews totales),
3. **todas** las reviews de los usuarios elegidos.

Preguntas: ¿cuánto historial queda por usuario?, ¿cuánto suma cruzar categorías?, ¿el groupby por usuario ya da un perfil?

Dos pasadas streaming por categoría (~10-20 min local con `All_Beauty` + `Amazon_Fashion`). Solo local.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

from proyecto_final_ceia.sampling import metadata_for_sampled_asins, user_quota_sample_reviews

CATEGORIES = ["All_Beauty", "Amazon_Fashion"]
QUOTA_PER_TIER = 2000
HASH_FRACTION = 0.25
SEED = 42
OUTPUT_DIR = Path("../data/samples_users")

In [2]:
t0 = time.time()
reviews_df, tiers_df = user_quota_sample_reviews(CATEGORIES, QUOTA_PER_TIER, HASH_FRACTION, SEED)
print(f"{len(reviews_df):,} reviews de {reviews_df['user_id'].nunique():,} usuarios en {time.time() - t0:.0f}s")
tiers_df

22,161 reviews de 6,000 usuarios en 97s


,users_population,users_selected,weight
tier,,,
1,535912,2000,267.9560
2-4,96543,2000,48.2715
5+,5623,2000,2.8115


## Historial por usuario

`users_population` es el tamaño del tier dentro de la población hasheada (25% de los usuarios). El peso permite volver a estadísticas poblacionales.

In [3]:
per_user = reviews_df.groupby("user_id").agg(
    tier=("user_tier", "first"),
    n_reviews=("rating", "size"),
    n_categories=("category", "nunique"),
    n_products=("parent_asin", "nunique"),
)
per_user.groupby("tier")[["n_reviews", "n_products"]].describe().round(1)

n_reviews                                       n_products             \
         count mean   std  min  25%  50%  75%    max      count mean   std   
tier                                                                         
1       2000.0  1.0   0.0  1.0  1.0  1.0  1.0    1.0     2000.0  1.0   0.0   
2-4     2000.0  2.3   0.6  2.0  2.0  2.0  2.0    4.0     2000.0  2.2   0.6   
5+      2000.0  7.8  10.8  5.0  5.0  6.0  8.0  375.0     2000.0  7.6  10.8   

                                 
      min  25%  50%  75%    max  
tier                             
1     1.0  1.0  1.0  1.0    1.0  
2-4   1.0  2.0  2.0  2.0    4.0  
5+    1.0  5.0  6.0  7.0  373.0

In [4]:
# ¿Cuánto suma cruzar categorías? Usuarios con reviews en más de una, y en qué tier caerían mirando solo su mejor categoría
per_user_cat = reviews_df.groupby(["user_id", "category"]).size().unstack(fill_value=0)
max_single_cat = per_user_cat.max(axis=1).reindex(per_user.index)
tier_single_cat = max_single_cat.map(lambda n: "1" if n == 1 else ("2-4" if n <= 4 else "5+"))

cross = pd.DataFrame({
    "usuarios": per_user.groupby("tier").size(),
    "%_multi_categoria": (per_user["n_categories"] > 1).groupby(per_user["tier"]).mean().mul(100).round(1),
    "%_baja_de_tier_con_1_cat": (tier_single_cat != per_user["tier"]).groupby(per_user["tier"]).mean().mul(100).round(1),
})
cross

,usuarios,%_multi_categoria,%_baja_de_tier_con_1_cat
tier,,,
1,2000,0.0,0.0
2-4,2000,27.8,18.8
5+,2000,53.8,25.4


## Prueba: agrupar por usuario (perfil)

Mismo groupby que en el 04, ahora con historial completo. Se agrega la metadata de los productos reseñados para tener atributos de contenido en el perfil.

In [5]:
t0 = time.time()
meta_df = pd.concat(
    [metadata_for_sampled_asins(c, set(reviews_df.loc[reviews_df["category"] == c, "parent_asin"])) for c in CATEGORIES],
    ignore_index=True,
)
print(f"metadata: {len(meta_df):,} productos en {time.time() - t0:.0f}s")

meta_flat = pd.DataFrame({
    "parent_asin": meta_df["parent_asin"],
    "price": meta_df["price"],
    "store": meta_df["store"],
    "average_rating": meta_df["average_rating"],
})
enriched = reviews_df.drop(columns=["images"]).merge(meta_flat, on="parent_asin", how="left", validate="many_to_one")
assert len(enriched) == len(reviews_df)
enriched["review_len"] = enriched["text"].str.len()
# Desvío del usuario respecto del rating promedio del producto: señal de "exigencia"
enriched["rating_vs_product"] = enriched["rating"] - enriched["average_rating"]

metadata: 19,558 productos en 42s


In [6]:
user_profile = enriched.groupby("user_id").agg(
    tier=("user_tier", "first"),
    n_reviews=("rating", "size"),
    n_categories=("category", "nunique"),
    n_stores=("store", "nunique"),
    mean_rating=("rating", "mean"),
    std_rating=("rating", "std"),
    mean_rating_vs_product=("rating_vs_product", "mean"),
    median_price=("price", "median"),
    pct_verified=("verified_purchase", "mean"),
    mean_review_len=("review_len", "median"),
    span_days=("timestamp", lambda s: (s.max() - s.min()) / 86_400_000),
)
# Cobertura de cada atributo del perfil por tier (% usuarios con valor no nulo)
user_profile.drop(columns="tier").notna().groupby(user_profile["tier"]).mean().mul(100).round(1).T

tier,1,2-4,5+
n_reviews,100.0,100.0,100.0
n_categories,100.0,100.0,100.0
n_stores,100.0,100.0,100.0
mean_rating,100.0,100.0,100.0
std_rating,0.0,100.0,100.0
mean_rating_vs_product,100.0,100.0,100.0
median_price,20.2,31.6,50.6
pct_verified,100.0,100.0,100.0
mean_review_len,100.0,100.0,100.0
span_days,100.0,100.0,100.0


In [7]:
user_profile.groupby("tier").median(numeric_only=True).round(2).T

tier,1,2-4,5+
n_reviews,1.00,2.00,6.00
n_categories,1.00,1.00,2.00
n_stores,1.00,2.00,5.00
mean_rating,5.00,4.50,4.20
std_rating,NaN,0.58,0.98
mean_rating_vs_product,0.30,0.30,0.26
median_price,18.99,16.99,16.80
pct_verified,1.00,1.00,1.00
mean_review_len,95.00,96.00,113.00
span_days,0.00,301.69,1029.59


In [8]:
# La muestra sobre-representa usuarios activos: sin pesos, las estadísticas no son poblacionales
pd.Series({
    "mean_rating_sin_pesos": enriched["rating"].mean(),
    "mean_rating_ponderado": np.average(enriched["rating"], weights=enriched["user_weight"]),
    "%_usuarios_tier_1_muestra": 100 * (per_user["tier"] == "1").mean(),
    "%_usuarios_tier_1_poblacion": 100 * tiers_df.loc["1", "users_population"] / tiers_df["users_population"].sum(),
}).round(3)

mean_rating_sin_pesos           4.087
mean_rating_ponderado           3.976
%_usuarios_tier_1_muestra      33.333
%_usuarios_tier_1_poblacion    83.988
dtype: float64

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
reviews_df.to_parquet(OUTPUT_DIR / "reviews_sample.parquet", index=False)
meta_df.to_parquet(OUTPUT_DIR / "meta_sample.parquet", index=False)
tiers_df.to_parquet(OUTPUT_DIR / "tiers.parquet")
print(f"-> {OUTPUT_DIR}")

-> ..\data\samples_users


## Conclusiones

- **El muestreo por usuario funciona y es barato**: 6.000 usuarios con 22.161 reviews en ~100s (dos pasadas sobre ~3,2M reviews), sin guardar el raw.
- **Cruzar categorías suma historial**: con `All_Beauty` + `Amazon_Fashion`, el 16% de los usuarios tiene ≥2 reviews (contra el 7,7% en `All_Beauty` sola). El 54% de los usuarios `5+` reseña en las dos categorías, y el 25% de ellos bajaría de tier mirando una sola.
- **La población sigue dominada por usuarios de 1 review**: el 84% está en el tier `1` y solo el 0,9% en `5+`. La cuota los pone en un tercio de la muestra cada uno, así que cualquier estadística poblacional necesita `user_weight` (rating medio: 4,09 sin pesos contra 3,98 ponderado).
- **El groupby por usuario ya da un perfil en los tiers `2-4` y `5+`**: rating medio y dispersión, desvío contra el rating del producto, stores distintas y rango temporal (mediana de ~1.000 días en `5+`). El precio solo aparece en el 20-50% de los usuarios.
- **Hay usuarios que reseñan el mismo `parent_asin` varias veces** (variantes): en `5+`, `n_products` puede quedar por debajo de `n_reviews`.
- **Siguiente prueba**: sumar más categorías (el historial cross-categoría crece con cada una) y definir el tier `1` como caso de cold-start, perfilado solo por el contenido del producto.